# Parallel processing of array data with `dask`

In this demo and exercise, we will showcase how we can process array data in parallel with `dask`. We start with a simple processing function, where all the information we need to process a single chunk is present in that chunk. Then we move on to a more complex case where, in order to process a chunk, we require data from neighbouring chunks.

In [ ]:
from course_large_array_data import monitor

We will reuse the data from the previous exercise, which we have conveniently stored on local disk.

In [ ]:
from pathlib import Path

import dask.array as da
import numpy as np

from course_large_array_data import print_in_raw_gb

in_path = Path("./data/twophoton_series.zarr")

single_neuron = da.from_zarr(in_path)
print(single_neuron.shape)
print_in_raw_gb(single_neuron)

average = da.from_zarr(Path("./data/average.zarr"))
print(average.shape)
print_in_raw_gb(average)

Note that the data is pretty small for this toy example. Normally, we'd just load it into a numpy array for ease. But here we are interested in showing how we can distribute calculations across chunks of a `dask` array, so we keep it lazy!

First, we find the pixels belonging to a single neuron by thresholding the average image, and store this in a mask. This does not do any parallel processing yet.

In [ ]:
%%monitor
import matplotlib.pyplot as plt

threshold = 0.4*average.max()
single_neuron_slice = (slice(140,150), slice(415,425))
print(threshold)
average_single_neuron = average[single_neuron_slice]
neuron_mask = average_single_neuron>threshold
plt.subplot(1, 2, 1)
plt.imshow(average_single_neuron)
plt.subplot(1, 2, 2)
plt.imshow(neuron_mask)
plt.show()


Next, we calculate the mean of the array over time, but only inside the mask. To do this, we 
* first set everything outside the mask to NaN ("Not a number")
* this allows us to then compute the mean of the not NaN entries in the array

In [ ]:
%%monitor

masked = da.where(neuron_mask, single_neuron, np.nan)
raw_fluorescence_over_time =  da.nanmean(masked, axis=(1, 2))


This is already set up to do some concurrency under the hood: `dask` array operations use multi-threading by default. Because the result is still a dask array, we need to `.compute` its result to reuse it later for plotting and comparisons.

In [ ]:
%%monitor

raw_fluorescence_over_time_numpy = raw_fluorescence_over_time.compute()

Let's plot the raw fluorescence:

In [ ]:
%%monitor

plt.figure()
plt.plot(raw_fluorescence_over_time_numpy)
plt.show()


We will now perform the same calculation again, but using concurrency explicitly! First, we define a function that computes the mean of an array chunk inside a given mask.

In [ ]:
import dask.array as da


def masked_spatial_mean(chunk: np.ndarray, mask) -> np.ndarray:
    masked = np.where(mask, chunk, np.nan)
    return np.nanmean(masked, axis=(1, 2))

Then, we apply this function to each chunk in the `dask` array using the `map_blocks` function. We compute the result, and check it matches our previous calculation.

In [ ]:
%%monitor

raw_fluorescence_over_time_explicit = single_neuron.map_blocks(masked_spatial_mean, mask=neuron_mask, drop_axis=(1,2), dtype=np.float32).compute()

assert np.allclose(raw_fluorescence_over_time_numpy, raw_fluorescence_over_time_explicit)

The main difficulty in interpreting the code above above lies in the `drop_axis=(1,2)` parameter. This signifies that for each chunk in the input, the spatial dimensions will collapse and result in a time-dimension-only chunk - but each chunk's result can be processed independently. For more complex operations, we are not so lucky, as we will see below.

A typical operation is to "normalise" the raw fluorescence array by subtracting a smooth baseline, which we also divide the result by. Let's encode this complex filtering operation into a Python function called `delta_f_over_f`.

In [ ]:
from scipy.ndimage import gaussian_filter1d, maximum_filter1d, minimum_filter1d


def delta_f_over_f(F, sigma=5, min_max_size=20):
    F0 = gaussian_filter1d(F, sigma=sigma, axis=0)
    F0 = minimum_filter1d(F0, size=min_max_size, axis=0)
    F0 = maximum_filter1d(F0, size=min_max_size, axis=0)
    return (F - F0) / F0

We can easily apply this function to our example `dask` array

In [ ]:
%%monitor
df_f_native = delta_f_over_f(raw_fluorescence_over_time).compute()


In [ ]:
plt.figure(figsize=(30, 2))
plt.plot(df_f_native)
plt.show()

If we want to evaluate the function lazily (i.e. chunk-by-chunk), we will need information from neighbour chunks. We need to tell a special version of `map_blocks` called `map_overlap` which neighbouring information we need via the `depth` parameter. In our case, the `depth` parameter depends on the parameters of our filters.

In [ ]:
%%monitor

sigma = 10
min_max_size = 50
gaussian_radius = int(4.0 * sigma + 0.5)
minmax_radius = min_max_size // 2 

depth = {0: gaussian_radius + 2 * minmax_radius, 1: 0, 2: 0}
df_f_parallel = raw_fluorescence_over_time.map_overlap(
    delta_f_over_f,    
    sigma=sigma,
    min_max_size=min_max_size,
    depth=depth,
    boundary='reflect')

We can compute this function using the default `dask` scheduler.

In [ ]:
%%monitor
df_f_parallel_threaded = df_f_parallel.compute(scheduler='threads') # default for dask arrays

Equally, we can do the same using `multiprocessing`

In [ ]:
%%monitor
df_f_parallel_processes = df_f_parallel.compute(scheduler='processes')

It doesn't seem to make much of a difference whether we use multi-threading or multi-processing, but both seem to be maybe 1.5-2 times as fast as the `dask` native operation.

In [ ]:
plt.figure(figsize=(30, 2))
plt.plot(df_f_parallel_threaded)
plt.show()

This plot is a bit dense. Let's just plot the first 1000 time points to get a closer look

In [ ]:
plt.figure(figsize=(30, 2))
plt.plot(df_f_parallel_threaded[0:1000])
plt.show()

This looks like a reasonable normalised fluorescence trace given our simple processing steps.

## Stretch exercise

Try making a plot where you vary the number of workers available to the `dask` scheduler and plot the time `map_blocks` takes (hint: you can use the `time` module from standard Python). Can you explain your result?

Alternatively, feel free to try out some of the concepts presented here on your own data, invent your own related stretch exercise or help others in the room.

## Key take-aways

In summary, we have learnt that:
* you can apply chunkwise operations concurrently on a dask array using `map_blocks`
  * this allows you to define custom operations for your chunks.
  * this can improve performance, but often `dask` default functions are quite efficient already.
* if you need information from neighbouring chunks, use `map_overlap`
  * you will need to think about the `depth` parameter
* by default, this `dask` Arrays operate concurrently using multi-threading (which is a sensible default, but you can specify to use multi-processing instead if you want)